# SageMaker Serverless ImageNet Classification Tutorial

This notebook demonstrates how to deploy an ImageNet classification model to SageMaker Serverless Inference.

## Prerequisites
- AWS credentials configured
- SageMaker execution role with appropriate permissions
- boto3 and sagemaker Python packages installed

## Setup

In [ ]:
# Install required packages (if not already installed)
!pip install -q sagemaker boto3 torch torchvision

In [ ]:
import json
import boto3
import sagemaker
from sagemaker.pytorch import PyTorchModel
from sagemaker.serverless import ServerlessInferenceConfig
import time

# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name
role = sagemaker.get_execution_role()  # Use SageMaker execution role

print(f"Region: {region}")
print(f"Role: {role}")

## Step 1: Configure Deployment

In [ ]:
# Configuration
model_name = "resnet50"  # Options: resnet50, resnet18, efficientnet_b0, densenet121
memory_size = 4096  # Memory in MB: 1024, 2048, 3072, 4096, 5120, 6144
max_concurrency = 10  # Maximum concurrent invocations (1-200)

# Generate unique endpoint name
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
endpoint_name = f"imagenet-{model_name}-{timestamp}"

print(f"Model: {model_name}")
print(f"Endpoint name: {endpoint_name}")
print(f"Memory: {memory_size} MB")
print(f"Max concurrency: {max_concurrency}")

## Step 2: Create and Deploy the Model

This step creates a PyTorch model and deploys it to a serverless endpoint. The deployment takes several minutes.

In [ ]:
# Create PyTorch Model
pytorch_model = PyTorchModel(
    model_data=None,  # Not needed for pretrained models
    role=role,
    entry_point="inference.py",
    framework_version="1.13.1",
    py_version="py39",
    env={
        "MODEL_NAME": model_name
    },
    sagemaker_session=sagemaker_session
)

# Configure serverless inference
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=memory_size,
    max_concurrency=max_concurrency,
)

print("Starting deployment...")
print("This will take several minutes. Please wait...")

In [ ]:
# Deploy the model (this takes several minutes)
predictor = pytorch_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name=endpoint_name
)

print("\n" + "="*60)
print("Deployment successful!")
print("="*60)
print(f"Endpoint name: {endpoint_name}")
print(f"Model: {model_name}")
print(f"Region: {region}")
print("="*60)

## Step 3: Prepare Test Image

Upload a test image to S3 and generate a presigned URL.

In [ ]:
# Option A: Upload a local image to S3
# Uncomment and modify the path below if you have a local image

# import os
# local_image_path = "path/to/your/image.jpg"  # Change this to your image path
# s3_bucket = sagemaker_session.default_bucket()
# s3_key = f"test-images/{os.path.basename(local_image_path)}"

# # Upload to S3
# s3_client = boto3.client('s3')
# s3_client.upload_file(local_image_path, s3_bucket, s3_key)
# print(f"Uploaded to s3://{s3_bucket}/{s3_key}")

# Option B: Use a sample image URL (public URL)
# For this example, we'll use a direct URL
# In production, use presigned URLs for private images

sample_image_url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
print(f"Using sample image: {sample_image_url}")

In [ ]:
# If you uploaded to S3, generate a presigned URL
# Uncomment this block if using S3

# s3_client = boto3.client('s3', region_name=region)
# presigned_url = s3_client.generate_presigned_url(
#     'get_object',
#     Params={'Bucket': s3_bucket, 'Key': s3_key},
#     ExpiresIn=3600  # 1 hour
# )
# print(f"Presigned URL generated (expires in 1 hour)")
# image_url = presigned_url

# For this demo, use the sample URL
image_url = sample_image_url

## Step 4: Invoke the Endpoint

Send a prediction request to the serverless endpoint.

In [ ]:
# Prepare payload
payload = {
    "url": image_url
}

# Invoke endpoint
print("Invoking endpoint...")
print(f"Image URL: {image_url}\n")

runtime_client = boto3.client('sagemaker-runtime', region_name=region)

response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=json.dumps(payload)
)

# Parse response
result = json.loads(response['Body'].read().decode())

print("="*60)
print("Prediction Results:")
print("="*60)

predictions = result.get('predictions', [])
for i, pred in enumerate(predictions, 1):
    class_name = pred['class']
    probability = pred['probability']
    print(f"{i}. {class_name:30s} {probability*100:6.2f}%")

print("="*60)

## Step 5: Multiple Predictions (Optional)

Test the endpoint with multiple images.

In [ ]:
# Test with multiple sample images
test_images = [
    "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg",
    # Add more test image URLs here
]

for idx, test_url in enumerate(test_images, 1):
    print(f"\nTest {idx}: {test_url}")
    
    payload = {"url": test_url}
    
    response = runtime_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=json.dumps(payload)
    )
    
    result = json.loads(response['Body'].read().decode())
    predictions = result.get('predictions', [])
    
    print(f"Top prediction: {predictions[0]['class']} ({predictions[0]['probability']*100:.2f}%)")

## Step 6: Cleanup

**Important**: Delete the endpoint to avoid ongoing charges.

In [ ]:
# Delete the endpoint
print(f"Deleting endpoint: {endpoint_name}")

sagemaker_client = boto3.client('sagemaker', region_name=region)
sagemaker_client.delete_endpoint(EndpointName=endpoint_name)

print("Endpoint deleted successfully")
print("Note: Endpoint config and model are also removed automatically")

## Summary

In this tutorial, you learned how to:
1. Deploy a PyTorch ImageNet model to SageMaker Serverless Inference
2. Configure serverless parameters (memory, concurrency)
3. Invoke the endpoint with presigned URLs
4. Get top-5 classification predictions
5. Clean up resources

### Next Steps
- Try different model architectures (ResNet18, EfficientNet, DenseNet)
- Adjust serverless configuration for your workload
- Integrate with your application using AWS SDK
- Monitor endpoint metrics in CloudWatch